# 面试问题：LLM 弹性分布式训练遇到 worker failure 或 membership change 时，怎样安全恢复？

**一句话回答。** 把 `torchrun`/Elastic Agent 当成进程生命周期控制面：所有节点用同一 `rdzv-id`、backend 和 endpoint 完成 rendezvous；任一 worker 失败或成员加入/离开时，停止全部存活 worker，重新形成 WorkerGroup，并从最后一个原子发布且校验完整的 checkpoint 整组恢复。训练逻辑不能把 `RANK`、`WORLD_SIZE` 或每 rank 数据偏移当成稳定身份，而应恢复拓扑无关的模型、优化器、调度器、混合精度、全局数据游标和随机性契约，再按新 world size 重分片。

本 Notebook 用标准库和小状态机模拟控制面，不真正启动多进程、GPU 或分布式通信。代码重点验证不稳定 rank、gang restart、一致 checkpoint、数据不重不漏以及 lost-work 预算。

**资料入口。** [PyTorch Elastic Run 官方文档](https://docs.pytorch.org/docs/stable/elastic/run) 说明：worker failure 会令所有 worker 停止并在 `max_restarts` 内重启；节点加入或离开也会重建 WorkerGroup，`RANK` 与弹性模式下的 `WORLD_SIZE` 都不能硬编码。

In [ ]:
question = "弹性分布式 LLM 训练怎样从 worker failure 安全恢复"  # 定义本 Notebook 对应的核心面试问题。
elastic_config = {"job_id": "llm-pretrain-7", "min_workers": 2, "max_workers": 4, "max_restarts": 3}  # 定义同一作业的弹性成员范围与最大重试次数。
assert "worker failure" in question  # 验证问题明确覆盖工作进程故障。
assert elastic_config["min_workers"] < elastic_config["max_workers"]  # 验证配置允许成员数量发生弹性变化。
assert elastic_config["max_restarts"] == 3  # 验证重启次数有明确上限。
assert elastic_config["job_id"].startswith("llm-")  # 验证 rendezvous 使用稳定作业标识而非临时 rank。

## 1. Rendezvous 只决定本轮成员和临时 rank

每次启动都以 job id 关联同一个训练作业，以 rendezvous backend/endpoint 协调“谁参加这一轮”。达到最小节点数后，控制面冻结本轮成员，产生新的 generation、rank map 和 world size。`RANK` 是本轮通信地址，不是持久 worker 身份；业务状态、数据游标和 checkpoint 文件名都不应只绑定 rank。下面用确定性排序模拟一次 rendezvous，真实系统则由后端完成租约、屏障和成员一致性。

In [ ]:
def rendezvous(job_id, generation, worker_ids, minimum, maximum):  # 模拟一次冻结成员并分配临时 rank 的 rendezvous。
    members = sorted(set(worker_ids))  # 对候选成员去重并固定本轮演示顺序。
    assert minimum <= len(members) <= maximum  # 验证当前成员数满足弹性配置边界。
    rank_map = {worker_id: rank for rank, worker_id in enumerate(members)}  # 为本轮成员生成从零连续的临时 rank。
    return {"job_id": job_id, "generation": generation, "members": members, "world_size": len(members), "rank_map": rank_map}  # 返回本轮不可混用的成员视图。
round_zero = rendezvous(elastic_config["job_id"], 0, ["worker-b", "worker-c", "worker-d"], 2, 4)  # 建立三个成员的初始训练轮次。
round_one = rendezvous(elastic_config["job_id"], 1, ["worker-a", "worker-b"], 2, 4)  # 模拟节点离开并有新节点加入后的轮次。
assert round_zero["world_size"] == 3  # 验证初始 world size 来自实际成员集合。
assert round_one["world_size"] == 2  # 验证成员变化后 world size 可以改变。
assert round_zero["rank_map"]["worker-b"] == 0  # 记录同一逻辑 worker 在初始轮次的临时 rank。
assert round_one["rank_map"]["worker-b"] == 1  # 验证重启后同一 worker 的 rank 可能变化。
assert set(round_one["rank_map"].values()) == {0, 1}  # 验证新一轮 rank 仍然连续且仅在本轮有效。

## 2. 故障与成员变化采用整组重启，而非幸存者偷偷续跑

同步数据并行中的梯度 collective 要求参与者集合一致。一个 worker 消失后，让其他 worker 跨过下一次 all-reduce 会挂起或产生错误语义。因此 Elastic Agent 会停止所有存活 worker，再让候选成员重新 rendezvous；scale-up/scale-down 同样形成新 WorkerGroup。训练入口必须幂等：启动后先查找并加载已提交 checkpoint，再初始化本轮进程组并继续训练。超过 `max_restarts` 时终止作业并保留故障证据。

In [ ]:
def restart_plan(active_group, failed_worker, next_worker_ids, restart_count, maximum_restarts):  # 根据故障和剩余预算生成整组重启计划。
    if restart_count >= maximum_restarts:  # 在重启预算耗尽时进入终止分支。
        return {"status": "terminated", "killed": sorted(active_group["members"]), "next_group": None}  # 保留整组停止语义并拒绝无限重试。
    next_group = rendezvous(active_group["job_id"], active_group["generation"] + 1, next_worker_ids, elastic_config["min_workers"], elastic_config["max_workers"])  # 为下一轮重新冻结成员和编号。
    return {"status": "restart", "cause": failed_worker, "killed": sorted(active_group["members"]), "next_group": next_group, "restart_count": restart_count + 1}  # 明确所有旧 worker 都会被停止。
failure_plan = restart_plan(round_zero, "worker-d", ["worker-a", "worker-b"], 0, 3)  # 模拟一个 worker 崩溃后的首次恢复。
terminal_plan = restart_plan(round_zero, "worker-d", ["worker-a", "worker-b"], 3, 3)  # 模拟重试次数已经耗尽。
assert failure_plan["status"] == "restart"  # 验证预算内故障进入重启流程。
assert set(failure_plan["killed"]) == set(round_zero["members"])  # 验证旧 WorkerGroup 全员被停止。
assert "worker-c" in failure_plan["killed"]  # 验证健康幸存者也不能绕过 gang restart。
assert failure_plan["next_group"]["generation"] == 1  # 验证新 WorkerGroup 使用新的 generation。
assert failure_plan["next_group"]["world_size"] == 2  # 验证新轮次采用重新协商的 world size。
assert terminal_plan["status"] == "terminated"  # 验证超过最大重试次数后作业终止。
assert terminal_plan["next_group"] is None  # 验证终止状态不会偷偷创建下一轮训练。

## 3. 一致 checkpoint 必须覆盖完整训练语义

只保存模型权重不足以恢复 LLM 训练。至少要保存模型参数、优化器动量/方差、学习率调度器、AMP scaler、全局 step/token、数据游标、随机状态、训练配置指纹、代码/数据版本和 schema version。分片文件应以 checkpoint generation 命名，manifest 记录每个必需组件的大小与哈希；恢复时先验证 schema 和配置兼容性，再重组或按新拓扑 reshard。教学示例把小字典编码为规范 JSON。

In [ ]:
import hashlib  # 导入标准库哈希函数以模拟分片完整性校验。
import json  # 导入标准库 JSON 以生成确定性的教学 checkpoint。
def canonical_bytes(value):  # 将教学状态编码为稳定字节序列。
    return json.dumps(value, sort_keys=True, separators=(",", ":"), ensure_ascii=False).encode("utf-8")  # 固定键顺序与分隔符以便重复计算哈希。
def checksum(payload):  # 计算单个 checkpoint 分片的内容摘要。
    return hashlib.sha256(payload).hexdigest()  # 返回十六进制 SHA-256 教学摘要。
training_state = {"model": {"layer.0.weight": [1.0, 2.0], "layer.1.weight": [3.0]}, "optimizer": {"layer.0.weight": {"m": 0.1}, "layer.1.weight": {"m": 0.2}}, "scheduler": {"step": 100, "lr": 0.0003}, "scaler": {"scale": 1024.0}, "progress": {"global_step": 100, "committed_cursor": 17}, "rng": {"base_seed": 2026, "python": "py-state-100", "cpu": "cpu-state-100", "cuda": ["cuda0-state-100"]}, "metadata": {"schema": 2, "config_hash": "cfg-a", "data_version": "corpus-v3"}}  # 构造覆盖训练语义的最小完整状态。
required_components = ("model", "optimizer", "scheduler", "scaler", "progress", "rng", "metadata")  # 声明恢复必须同时存在的组件集合。
checkpoint_shards = {name: canonical_bytes(training_state[name]) for name in required_components}  # 将每个逻辑组件编码成独立分片。
checkpoint_hashes = {name: checksum(payload) for name, payload in checkpoint_shards.items()}  # 为所有分片生成 manifest 所需哈希。
assert set(checkpoint_shards) == set(required_components)  # 验证 checkpoint 没有遗漏必需组件。
assert training_state["progress"]["global_step"] == 100  # 验证全局训练步数被纳入恢复状态。
assert training_state["rng"]["base_seed"] == 2026  # 验证随机性契约被纳入恢复状态。
assert len(checkpoint_hashes["model"]) == 64  # 验证 SHA-256 摘要具有预期长度。
assert checkpoint_hashes["model"] != checkpoint_hashes["optimizer"]  # 验证不同分片不会被误认为同一内容。

## 4. 原子 manifest 是 checkpoint 的提交记录

各 rank 先写 generation 专属的临时对象，完成 flush、校验与 barrier 后，由协调者发布包含全部分片哈希的不可变 manifest，最后原子更新 `LATEST` 指针。读者只跟随已提交 manifest；某个 worker 在上传一半时死亡，残留 staging 文件不能成为恢复点。对象存储没有目录 rename 时，可用唯一对象名、条件写入或 compare-and-swap 发布小 manifest。旧 generation 保留到新 generation 被验证和满足回收策略后再删除。

In [ ]:
object_store = {}  # 创建内存对象存储以模拟 staging、manifest 与 LATEST 指针。
def stage_generation(store, generation, payloads):  # 将一个 generation 的候选分片写入独立暂存前缀。
    for name, payload in payloads.items():  # 逐个写入本轮已经完成的分片。
        store[f"staging/{generation}/{name}"] = payload  # 使用 generation 隔离并发或失败写入。
def publish_generation(store, generation, required_names, expected_hashes):  # 校验全部分片后原子发布小 manifest。
    missing = [name for name in required_names if f"staging/{generation}/{name}" not in store]  # 收集尚未成功落盘的必需组件。
    if missing:  # 在任何分片缺失时拒绝推进提交指针。
        return {"committed": False, "missing": missing}  # 返回可审计失败原因并保留旧恢复点。
    actual_hashes = {name: checksum(store[f"staging/{generation}/{name}"]) for name in required_names}  # 从持久对象重新计算所有哈希。
    if actual_hashes != expected_hashes:  # 在上传损坏或版本混搭时拒绝发布。
        return {"committed": False, "missing": [], "corrupt": True}  # 标记完整性失败且不更新 LATEST。
    manifest = {"generation": generation, "required": list(required_names), "hashes": actual_hashes}  # 构造不可变提交记录。
    store[f"manifest/{generation}"] = canonical_bytes(manifest)  # 先持久化可以独立验证的 manifest。
    store["LATEST"] = str(generation)  # 最后更新唯一的小指针以完成逻辑原子提交。
    return {"committed": True, "manifest": manifest}  # 返回成功发布的 manifest 供审计。
stage_generation(object_store, 12, checkpoint_shards)  # 模拟完整写入 generation 12 的所有分片。
commit_twelve = publish_generation(object_store, 12, required_components, checkpoint_hashes)  # 发布第十二代一致 checkpoint。
torn_shards = {name: payload for name, payload in checkpoint_shards.items() if name != "optimizer"}  # 模拟故障导致 optimizer 分片未上传。
stage_generation(object_store, 13, torn_shards)  # 写入不完整的第十三代暂存对象。
commit_thirteen = publish_generation(object_store, 13, required_components, checkpoint_hashes)  # 尝试发布不完整 checkpoint。
assert commit_twelve["committed"] is True  # 验证完整 generation 可以被提交。
assert object_store["LATEST"] == "12"  # 验证当前可恢复指针仍指向最后完整 generation。
assert commit_thirteen["committed"] is False  # 验证残缺 generation 不能成为恢复点。
assert commit_thirteen["missing"] == ["optimizer"]  # 验证失败原因精确指出缺少优化器状态。
assert "manifest/13" not in object_store  # 验证拒绝发布时没有伪造 manifest。

## 5. 恢复时先验证一致性，再适配新的 world size

加载端读取 `LATEST`，逐个核对必需分片与哈希，任何缺失或损坏都回退到更早的完整 generation，而不是混搭两个 step。若 world size 改变，checkpoint 必须支持拓扑无关的逻辑参数名或分布式 checkpoint 重分片；旧 rank 的 optimizer shard 不能直接交给同编号的新 rank。下面用参数名哈希的简化分配展示：所有逻辑状态保持一次且仅一次，但归属会随 world size 改变。

In [ ]:
def read_latest_committed(store):  # 从原子指针加载并完整验证最后一次提交。
    generation = int(store["LATEST"])  # 读取唯一已发布 generation 而不是扫描暂存目录。
    manifest = json.loads(store[f"manifest/{generation}"].decode("utf-8"))  # 解码该 generation 的不可变 manifest。
    payloads = {name: store[f"staging/{generation}/{name}"] for name in manifest["required"]}  # 严格按 manifest 枚举必需分片。
    valid = all(checksum(payloads[name]) == manifest["hashes"][name] for name in manifest["required"])  # 重新校验每个分片的内容摘要。
    return {"generation": generation, "manifest": manifest, "payloads": payloads, "valid": valid}  # 返回可用于恢复的验证结果。
def reshard(parameter_names, world_size):  # 按新 world size 重新分配逻辑参数状态。
    return {rank: [name for index, name in enumerate(sorted(parameter_names)) if index % world_size == rank] for rank in range(world_size)}  # 确保每个参数恰好归属一个新 rank。
loaded = read_latest_committed(object_store)  # 加载最后一个已提交而非最后一个暂存 generation。
parameter_names = list(json.loads(loaded["payloads"]["model"].decode("utf-8")))  # 从逻辑模型状态取得稳定参数名。
old_owners = reshard(parameter_names, 3)  # 模拟 checkpoint 写入时的三个 rank 分片归属。
new_owners = reshard(parameter_names, 2)  # 模拟恢复后改为两个 rank 的重分片归属。
assert loaded["generation"] == 12  # 验证恢复自动忽略残缺的 generation 13。
assert loaded["valid"] is True  # 验证所加载分片全部通过哈希检查。
assert set(sum(old_owners.values(), [])) == set(parameter_names)  # 验证旧拓扑覆盖全部逻辑参数。
assert set(sum(new_owners.values(), [])) == set(parameter_names)  # 验证新拓扑仍覆盖全部逻辑参数。
assert len(sum(new_owners.values(), [])) == len(set(sum(new_owners.values(), [])))  # 验证重分片后没有重复参数归属。
assert old_owners != new_owners  # 验证不能把旧 rank 的物理分片原样映射给新 rank。
assert set(json.loads(loaded["payloads"]["optimizer"].decode("utf-8"))) == set(parameter_names)  # 验证优化器状态可按稳定参数名重新关联。

## 6. 数据恢复使用全局提交游标或稳定 sample id

每 rank 保存“我读到第几个 batch”会把旧 world size 烙进 checkpoint，重新分组后容易重复或跳过样本。更稳妥的契约是持久化全局已提交 token/sample 游标、epoch 与 sampler seed，或者使用可重放的稳定 sample id；恢复后基于新 world size 重新划分剩余集合。预取但尚未完成 optimizer step 的数据不能推进提交游标，所以故障后允许它被重放。若数据源只能至少一次投递，还要在指标与副作用层用 step/sample id 去重。

In [ ]:
def assign_remaining(total_samples, committed_cursor, world_size):  # 根据全局提交游标和新 world size 划分剩余样本。
    remaining = list(range(committed_cursor, total_samples))  # 只排除 checkpoint 已确认完成的样本前缀。
    return {rank: remaining[rank::world_size] for rank in range(world_size)}  # 用新 rank 对剩余稳定 sample id 做互斥切片。
def flatten_assignments(assignments):  # 汇总所有 rank 获得的样本以便验证全局语义。
    return [sample_id for samples in assignments.values() for sample_id in samples]  # 将各 rank 子序列拼成一个检查列表。
committed_cursor = training_state["progress"]["committed_cursor"]  # 从一致 checkpoint 恢复全局提交游标。
before_failure_prefetch = [17, 18]  # 模拟故障前已经预取但尚未提交 optimizer step 的样本。
assignment_two = assign_remaining(24, committed_cursor, 2)  # 按恢复后的两个 rank 重新划分剩余数据。
assignment_three = assign_remaining(24, committed_cursor, 3)  # 展示另一种 world size 下的等价全局语义。
flat_two = flatten_assignments(assignment_two)  # 汇总两个 rank 的恢复分配。
flat_three = flatten_assignments(assignment_three)  # 汇总三个 rank 的恢复分配。
assert set(flat_two) == set(range(17, 24))  # 验证两个 rank 不跳过任何未提交样本。
assert set(flat_three) == set(range(17, 24))  # 验证三个 rank 也不跳过任何未提交样本。
assert len(flat_two) == len(set(flat_two))  # 验证新一轮分配不会让两个 rank 重复处理样本。
assert len(flat_three) == len(set(flat_three))  # 验证不同 world size 下仍保持互斥归属。
assert before_failure_prefetch[0] in flat_two  # 验证预取但未提交的样本会在故障后安全重放。
assert 16 not in flat_two  # 验证 checkpoint 已提交完成的样本不会再次训练。
assert assignment_two != assignment_three  # 验证物理 rank 分配可变而全局样本集合保持不变。

## 7. RNG 要区分精确恢复与拓扑变化后的统计复现

固定拓扑恢复时应还原 Python、CPU、各 CUDA device、dropout 与 sampler 的 RNG 状态。world size 改变后，collective 顺序、batch 构成和浮点归约次序都可能改变，通常不能承诺逐 bit 相同。工程上要明确复现等级：相同拓扑追求精确恢复；弹性拓扑可用 base seed、global step、稳定 sample id 和逻辑随机流派生 counter seed，使同一样本的增强/遮罩不依赖临时 rank，同时接受数值轨迹可能分叉。

In [ ]:
def counter_seed(base_seed, global_step, sample_id, logical_stream):  # 为稳定样本和逻辑随机流派生拓扑无关种子。
    material = f"{base_seed}|{global_step}|{sample_id}|{logical_stream}".encode("utf-8")  # 将恢复所需的稳定坐标编码为字节。
    return int(hashlib.sha256(material).hexdigest()[:16], 16)  # 截取摘要得到可重复的教学整数种子。
def ephemeral_process_seed(base_seed, generation, rank):  # 为只属于本轮进程的随机流派生临时种子。
    return counter_seed(base_seed, generation, rank, "process")  # 显式把不稳定 generation 和 rank 限制在进程级用途。
base_seed = training_state["rng"]["base_seed"]  # 从 checkpoint 读取全局随机种子。
sample_seed_on_rank_zero = counter_seed(base_seed, 100, 19, "dropout")  # 模拟样本十九被 rank 零处理时的 dropout 种子。
sample_seed_on_rank_one = counter_seed(base_seed, 100, 19, "dropout")  # 模拟同一样本重分配到 rank 一后的 dropout 种子。
assert sample_seed_on_rank_zero == sample_seed_on_rank_one  # 验证样本随机性不依赖临时 rank。
assert counter_seed(base_seed, 100, 19, "dropout") != counter_seed(base_seed, 100, 20, "dropout")  # 验证不同样本获得不同随机流。
assert counter_seed(base_seed, 100, 19, "dropout") != counter_seed(base_seed, 100, 19, "mask")  # 验证不同逻辑用途不会意外共用随机流。
assert ephemeral_process_seed(base_seed, 0, 0) != ephemeral_process_seed(base_seed, 1, 0)  # 验证新 generation 的进程级种子可以更新。
assert "python" in training_state["rng"] and "cuda" in training_state["rng"]  # 验证固定拓扑精确恢复所需 RNG 类别被记录。

## 8. checkpoint 周期由 lost-work SLO 与写入开销共同决定

最坏丢失计算约等于 checkpoint 间隔乘单步耗时，平均可粗估为一半；恢复时间还包括故障探测、停止进程、rendezvous、加载和重新编译/预热。间隔太长违反 lost-work 预算，太短则 checkpoint 开销过高。应测量真实 p95 写入与恢复耗时，求出“开销要求给出的最小间隔”和“丢失预算给出的最大间隔”，若区间为空就必须异步写入、增量 checkpoint、提高带宽或放宽 SLO，而不是随意选一个数字。

In [ ]:
import math  # 导入标准库数学函数以计算离散 checkpoint 周期。
def choose_checkpoint_interval(lost_work_budget_seconds, step_seconds, checkpoint_seconds, maximum_overhead):  # 联合 lost-work SLO 与写入开销求可行周期。
    minimum_steps = math.ceil(checkpoint_seconds * (1.0 - maximum_overhead) / (maximum_overhead * step_seconds))  # 计算满足最大开销比例所需的最小步数。
    maximum_steps = math.floor(lost_work_budget_seconds / step_seconds)  # 计算满足最坏丢失预算允许的最大步数。
    feasible = minimum_steps <= maximum_steps  # 判断两个约束是否存在交集。
    chosen_steps = maximum_steps if feasible else None  # 在可行时选择丢失预算允许的最长周期以减少写入。
    return {"minimum_steps": minimum_steps, "maximum_steps": maximum_steps, "chosen_steps": chosen_steps, "feasible": feasible}  # 返回决策及可审计边界。
checkpoint_policy = choose_checkpoint_interval(120.0, 2.0, 8.0, 0.10)  # 为两分钟丢失预算和百分之十写入开销求周期。
chosen_seconds = checkpoint_policy["chosen_steps"] * 2.0  # 将选定步数转换为最坏丢失计算秒数。
observed_overhead = 8.0 / (chosen_seconds + 8.0)  # 估算该周期下 checkpoint 时间占比。
recovery_time = 20.0 + 30.0 + 40.0  # 汇总故障探测、rendezvous 与 checkpoint 加载时间。
assert checkpoint_policy["feasible"] is True  # 验证当前 SLO 与开销约束存在共同可行解。
assert checkpoint_policy["minimum_steps"] == 36  # 验证开销上限推导出的最小间隔。
assert checkpoint_policy["maximum_steps"] == 60  # 验证 lost-work 预算推导出的最大间隔。
assert chosen_seconds <= 120.0  # 验证最坏丢失计算不超过业务预算。
assert observed_overhead <= 0.10  # 验证选定周期满足 checkpoint 开销目标。
assert recovery_time == 90.0  # 验证恢复时间与丢失计算采用两个独立指标。
assert choose_checkpoint_interval(20.0, 2.0, 20.0, 0.05)["feasible"] is False  # 验证约束冲突时系统会显式报告不可行。

## 面试总结

完整回答可以沿 **rendezvous → 整组停止 → 重新编号 → 读取原子 manifest → 校验完整分片 → 按新拓扑重分片 → 从全局数据游标和随机性契约恢复 → 受 lost-work 预算约束继续训练** 展开。

面试中还应主动说清五个边界：第一，Elastic 负责发现、编排和重启，不自动替训练代码保存状态；第二，rank/world size 都是不稳定运行时属性；第三，checkpoint 的一致性以 manifest 提交为准；第四，membership change 后通常只能保证语义连续而非 bitwise 一致；第五，必须演练 worker crash、节点扩缩、上传中断、分片损坏、旧 schema、游标重放以及超过最大重试次数。生产实现还需结合真实分布式 checkpoint、对象存储条件写、监控告警和故障注入。